In [1]:
import numpy as np

# Lab 08: Variational Inference

In this session, we explore how to use Variational Inference in two different problems: to infer the parameter of a simple decision model with fixed-form VI, and to infer the parameters of a univariate normal distribution.

## Part 1: Fixed-form VI for a simple decision model

We consider a problem where a user has to make a binary decision based on some characteristics $x$. The decision, denoted by $y$, is either to accept or to reject $x$. We consider a unique user with a fixed decision policy (no evolution over time). We observe several decisions of the user, combined into a dataset $\mathcal{D} = \lbrace (x_1, y_1), \dotsc, (x_N, y_N) \rbrace$.

### Probabilistic model

We model this process as follows:
* Likelihood: $p(y_n = 1 \, | \, x_n, z) = \frac{1}{1 + \exp(- z x_n)}$
* Prior: $p(z \, | \, \mu, \sigma^2) = \mathcal{N}(z ; \mu, \sigma^2)$

In this model, $z$ is a real-valued latent parameter which indicates the tendency of the user to accept or reject the decisions. We can see that a high $z$ corresponds to a tendency to accept, while a low $z$ corresponds to a tendency to reject. Note that this corresponds to a Bernoulli distribution, the parameter of which depends on $x_n$. 

#### Question 1

Give an expression of $\log p(y_n \, | \, x_n, z)$ for any $y_n$ and implement it.

### <font color='green'><u>Solution</u></font>

We have 

$$
\log p(y_n = 1\mid x_n, z) = \log \frac{1}{1 + \exp(-zx_n)} = \underbrace{\log 1}_{=\,0} - \log(1 + \exp(-z x_n))
$$ 

and

$$
\log p(y_n = 0 \mid x_n, z) = \log \frac{1 + \exp(-zx_n) - 1}{1 + \exp(-zx_n)} = -zx_n - \log(1 + \exp(-zx_n)).
$$

In [2]:
def log_likelihood(y, x, z):
    a = - z * x
    return (1-y) * a - np.log(1 + np.exp(a))

#### Question 2

Express the ELBO $\text{L}(\mu, \sigma^2, \psi \, | \, \mathbf{X}, \mathbf{y})$ for a generic variational distribution $q_{\psi}$.

### <font color='green'><u>Solution</u></font>
\begin{align*}
\text{L}(\mu, \sigma^2, \psi \mid \mathbf{X}, \mathbf{y}) &= \mathbb E_{q_\psi(z)}[\log p(\mathbf{y}, z \mid \mu, \sigma^2, \mathbf{X}) - \log q_\psi(z)] = \mathbb E_{q_\psi(z)}\left[\log \left( p(z \mid \mu, \sigma^2) \prod_{n=1}^N p(y_n \mid z, \mathbf{x}_n)\right) - \log q_\psi(z)\right] \\
&= \mathbb E_{q_\psi(z)}[\log p(z \mid \mu, \sigma^2)] + \sum_{n=1}^N \mathbb E_{q_\psi(z)}[\log p(y_n \mid z, \mathbf x_n)] - \mathbb E_{q_\psi(z)}[\log q_\psi(z)] \\
&= \sum_{n=1}^N \mathbb E_{q_\psi(z)}[\log p(y_n \mid z, \mathbf x_n)] - D_\mathrm{KL}(q_\psi(z) \Vert p(z \mid \mu, \sigma^2)).
\end{align*}

<hr>

We will now use variational inference (VI) to infer the posterior over the latent variable $z$. Note that the parameters $\mu, \sigma^2$ are the parameters of the prior and supposed to be known. We will not consider the inference of these parameters in this session. 



### Reminder: Laplace approximation
 
We first consider that the variational distribution is a normal distribution:
$$
q_{m, s^2}(z) = \mathcal{N}(z; m, s^2)
$$
This choice of a variational distribution is called Laplace approximation. 

The goal is now to use a gradient-based method to optimize the ELBO with respect to $m$ and $s^2$. A difficulty here is that the gradient cannot be computed in closed-form, so we need to use an approximation. For this, we use Monte-Carlo sampling. 


### Monte-Carlo Sampling

The base idea behind Monte-Carlo sampling is to estimate an expectation $\mathbb{E}[f(X)]$ by first sampling $J$ i.i.d. values from random variable $X$, from which we have:
$$
\mathbb{E}[f(X)] \simeq \frac{1}{J} \sum_{j=1}^J f(x_j)
$$
With $f(x) = x$, this corresponds to the estimation of the mean of a sample. 

We illustrate this idea with a simple example. Let $X_1$ and $X_2$ be two random variables following a normal distribution of respective parameters $(\mu_1, \sigma_1)^2$ and $(\mu_2, \sigma_2)^2$. It can be verified (and this is a good exercise) that $X_1 + X_2$ follows a normal distribution of mean $\mu_1 + \mu_2$ and of variance $\sigma_1^2 + \sigma_2^2$.


#### Question 3

Implement a Monte-Carlo sampling of $X_1 + X_2$ (where the parameters of $X_1$ and $X_2$ can be chosen arbitrarily) and verify that it indeed converges to $\mathbb{E}[X_1 + X_2] = \mu_1 + \mu_2$.

### <font color='green'><u>Solution</u></font>

In [3]:
def MC_sampling_sum_gaussians(mu1, sigma1, mu2, sigma2, n_samples):
    x1 = np.random.normal(loc=mu1, scale=sigma1, size=n_samples)
    x2 = np.random.normal(loc=mu2, scale=sigma2, size=n_samples)
    return (x1 + x2).mean()

# Choice of the normal distributions:
mu1 = 2
sigma1 = 0.5
mu2 = 3
sigma2 = 0.2

n_samples = 100

print("MC estimation:", MC_sampling_sum_gaussians(mu1, sigma1, mu2, sigma2, n_samples))
print("Real value:", mu1 + mu2)

MC estimation: 4.991743393046437
Real value: 5


### Reparametrization trick

To use a gradient descent, we need to compute the gradient of the ELBO with respect to $\psi = (m, s^2)$. We would like to use Monte-Carlo sampling for this, but this is not possible, $\nabla_\psi \mathbb{E}_{Z \sim q_\psi}[f(Z)] \neq \mathbb{E}_{Z \sim q_\psi}[\nabla_\psi f(Z)]$. We solve this problem by using the reparametrization trick, which consists in changing the variable in the expectation to make its distribution independ from $\psi$. 

#### Question 4

Show that:
$$
\mathbb{E}_{Z \sim \mathcal{N}(m, s^2)}[f(Z)] = \mathbb{E}_{Z^\prime \sim \mathcal{N}(0, 1)}[f(\mu + s Z^\prime)]
$$

### <font color='green'><u>Solution</u></font>
It suffices to show that if $Z^\prime \sim \mathcal N(0, 1)$, then $Z := \mu + Z^\prime \sigma \sim \mathcal N(\mu, \sigma^2)$. Since $Z$ is obtained from $Z^\prime$ through the application of a transform $\phi_{\mu,\sigma^2}(z^\prime) = \mu + z^\prime \sigma$, we can apply integration by substitution. The inverse transformation is $\phi_{\mu,\sigma^2}^{-1}(z) = (z - \mu) / \sigma$ and has derivative $\frac{\mathrm d \phi_{\mu,\sigma^2}^{-1}}{\mathrm d z} = \sigma^{-1}$. Hence,
$$p_Z(z) = \left\lvert \frac{\mathrm d \phi_{\mu,\sigma^2}^{-1}}{\mathrm d z} \right\rvert p_{Z^\prime}(\phi_{\mu,\sigma^2}^{-1}(z)) = \frac{1}{\sigma} \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{((z - \mu) / \sigma)^2}{2}\right),$$
which is just the PDF of $\mathcal N(\mu, \sigma^2)$.

#### Question 5 (optional)

Using the result of Question 4 and noticing that it is now possible to permute the gradient and the expectation, implement a Monte-Carlo sampling of the gradient of the ELBO. You can either compute the gradient manually or using automatic tools.

### <font color='green'><u>Solution</u></font>
Ignoring the KL-term, one can analytically compute the gradient of the log-likelihood to be
$$\nabla_{(\mu,\sigma)} \log p(y_n \mid x_n, z) = x_n \left(\frac{\exp(-zx_n)}{1 + \exp(-zx_n)} - (1 - y_n)\right) \begin{pmatrix} 1 \\ \varepsilon \end{pmatrix}.$$

In [5]:
def log_likelihood_grad(y, x, z, eps):
    tmp = - z * x
    tmp2 = np.exp(-tmp)
    tmp3 = x * (tmp2 / (1 + tmp2) - (1 - y))  # (N,L)
    res = tmp3 * np.stack((np.ones_like(eps), eps))
    return res.mean(axis=1)  # mean along L-axis

z_true = -0.7
N = 100  # Sample size
xs = np.random.randn(N) * 2 + 2
ys = np.random.binomial(1, 1 / (1 + np.exp(-z_true * xs)), size=(N,))

mu, sigma = 0, 1  # Initial guess

L = 50  # Monte Carlo samples

# Reparameterization trick
eps = np.random.randn(N, L)
zs = mu + eps * sigma

# [:, None] appends an empty axis for the "Monte Carlo dimension"
llh_grad = log_likelihood_grad(xs[:, None], ys[:, None], zs, eps)
llh_grad

array([[-0.18290451, -0.20205928, -0.18064484, -0.19272038, -0.18550825,
        -0.15661847, -0.19148484, -0.20791593, -0.20138029, -0.1814657 ,
        -0.17015495, -0.19815954, -0.18416961, -0.18634617, -0.19253582,
        -0.19816367, -0.18479763, -0.20834635, -0.19734101, -0.16288261,
        -0.16195904, -0.18261383, -0.17522139, -0.1807099 , -0.19288863,
        -0.18739114, -0.17639019, -0.21400341, -0.19178296, -0.21289059,
        -0.17895524, -0.18408578, -0.17208191, -0.19110153, -0.19435344,
        -0.17643639, -0.19613964, -0.16440158, -0.16848021, -0.17296759,
        -0.18318853, -0.18887332, -0.18362478, -0.17031875, -0.18730292,
        -0.19814475, -0.17888813, -0.1844785 , -0.17677377, -0.17731585],
       [ 0.11838371,  0.26956372,  0.05928383, -0.0786035 , -0.01026549,
         0.03034115,  0.09005852,  0.05935788,  0.27964329,  0.08195915,
         0.17959918,  0.2069196 , -0.10306612,  0.15075673,  0.09076255,
         0.3162161 ,  0.01001573,  0.19454287,  0.

## Part 2: Free-form VI

We assume a dataset $\mathcal{D} = \lbrace x_1, \dotsc, x_N \rbrace$ and consider the following model:
* Likelihood: Normal distribution of mean $\mu$ and precision $\lambda = 1 / \sigma^2$:
$$
p(\mathcal{D} \, | \, \theta) = \prod_{n=1}^N \mathcal{N}(x_n \, | \, \mu, \lambda^{-1})
$$
* Prior: Conjugate prior for the normal distribution: 
$$
p(\mu, \lambda) = \text{Gamma}(\lambda \, | \, a_0, b_0) \, \mathcal{N}(\mu \, | \, \mu_0, (\kappa_0 \lambda)^{-1})
$$

We use the mean field approximation: 
$$
q(\mu, \lambda) = q_{\mu}(\mu | \psi_\mu) q_{\lambda}(\lambda | \psi_\lambda)
$$

#### Question 6

Compute the log probability for joint distribution $\log p(\mu, \lambda, \mathcal{D})$.

### <font color='green'><u>Solution</u></font>
\begin{align*}
\log p(\mu, \lambda, \mathcal{D}) &= \log(p(\lambda) p(\mu \mid \lambda) p(\mathcal D \mid \mu, \lambda)) \\
&= \log \mathrm{Gamma}(\lambda; a_0, b_0) + \log \mathcal N(\mu; \mu_0, (\kappa_0 \lambda)^{-1}) + \sum_{n=1}^N \log \mathcal N(x_n; \mu, \lambda^{-1}) \\
&= \log \left(\frac{b_0^{a_0}}{\Gamma(a_0)} \lambda^{a_0-1} e^{-b_0 \lambda}\right) + \log \left( \frac{1}{\sqrt{2\pi(\kappa_0 \lambda)^{-1}}}\exp\left(-\frac{(\mu - \mu_0)^2}{2(\kappa_0 \lambda)^{-1}}\right) \right)  + \sum_{n=1}^N \log \left( \frac{1}{\sqrt{2\pi\lambda^{-1}}} \exp\left(-\frac{(x_n - \mu)^2}{2\lambda^{-1}}\right) \right)\\
&= \textcolor{red}{\log\frac{b_0^{a_0}}{\Gamma(a_0)}} + (a_0-1)\log\lambda - b_0 \lambda + \frac{1}{2} (\log\lambda + \textcolor{red}{\log\kappa_0 - \log(2\pi)}) - \frac{\kappa_0 \lambda}{2}(\mu - \mu_0)^2 + \frac{N}{2}(\log \lambda - \textcolor{red}{\log(2\pi)}) - \frac{\lambda}{2} \sum_{n=1}^N (x_n - \mu)^2 \\
&= (a_0-1)\log\lambda - b_0 \lambda + \frac{1}{2} \log\lambda - \frac{\kappa_0 \lambda}{2}(\mu - \mu_0)^2 + \frac{N}{2} \log \lambda - \frac{\lambda}{2} \sum_{n=1}^N (x_n - \mu)^2 + \textcolor{red}{\mathrm{const}}.
\end{align*}

#### Question 7

Show that $q_{\mu}(\mu | \psi_\mu)$ is a normal distribution. Express its mean and variance.

### <font color='green'><u>Solution</u></font>
Insert the result of Question 6 in the expectation and observe that terms not involving $\mu$ are constants:
\begin{align*}
\log q_\mu(\mu \mid \psi_\mu) = \mathbb E_{q_\lambda(\cdot \mid \psi_\lambda)}[\log p(\mu, \lambda, \mathcal D)]
= \mathbb E_{q_\lambda(\cdot \mid \psi_\lambda)}\left[-\frac{\kappa_0\lambda}{2}(\mu - \mu_0)^2 - \frac{\lambda}{2} \sum_{n=1}^N (x_n - \mu)^2\right]
= - \frac{\mathbb E_{q_\lambda(\cdot \mid \psi_\lambda)}[\lambda]}{2}\left(\kappa_0(\mu - \mu_0)^2 + \sum_{n=1}^N (x_n - \mu)^2\right).
\end{align*}

With some more rearrangement, we can see that this is the logarithm of a normal distribution with parameters $\mu_N = \frac{\kappa_0\mu_0 + N \overline x}{\kappa_0 + N}$ and $\kappa_N = (\kappa_0 + N)\mathbb E_{q\lambda(\cdot \mid \psi_\lambda)}[\lambda]$.

#### Question 8 

Show that $q_{\lambda}(\lambda | \psi_\lambda)$ is a Gamma distribution. Express its parameters.

### <font color='green'><u>Solution</u></font>
See slide 38 of lecture 8.

#### Question 9

Use the two obtained distributions to compute the missing parameters (the expectation terms in the parameters you have).

### <font color='green'><u>Solution</u></font>
See slide 39 of lecture 8.